## Setup 

In [106]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

TRAIN_PATH = '/kaggle/input/datasets/neeldipeshshah/ml-task-event-attendance-predictor/event_attendance_real_world - event_attendance_real_world.csv.csv'
TEST_PATH  = '/kaggle/input/datasets/neeldipeshshah/ml-task-event-attendance-predictor/event_attendance_test_real_world - event_attendance_test_real_world.csv.csv'

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

print('train shape:', train_raw.shape)
print('test  shape:', test_raw.shape)
train_raw.head()


train shape: (508, 10)
test  shape: (100, 9)


,student_id,event_type,registration_days_before,previous_events_registered,previous_events_attended,club_member,event_day,event_time,travel_distance_km,attended
0,S1080,Workshop,3.0,6.0,5.0,Yes,Monday,16:00,3.8,1.0
1,S1317,Hackathon,9.0,3.0,2.0,Yes,Saturday,14:00,5.6,0.0
2,S1486,Social,0.0,7.0,5.0,Yes,Sunday,14:00,3.9,1.0
3,S1397,Competition,10.0,0.0,0.0,Yes,Monday,18:00,10.1,1.0
4,S1168,Workshop,6.0,5.0,4.0,Yes,Monday,11:00,0.7,1.0


In [107]:

def quality_report(df, name):
    print('='*70)
    print(f'QUALITY REPORT: {name}   shape={df.shape}')
    print('='*70)

    miss = df.isnull().sum()
    miss = miss[miss > 0]
    print('\nMissing values:')
    print(miss if len(miss) else '  none')

    print('\nExact duplicate rows:', df.duplicated().sum())
    if 'student_id' in df.columns:
        dup_ids = df['student_id'].duplicated().sum()
        print('Duplicate student_id rows:', dup_ids)

    print('\nDtypes:')
    print(df.dtypes)

quality_report(train_raw, 'TRAIN (raw)')


QUALITY REPORT: TRAIN (raw)   shape=(508, 10)

Missing values:
event_type                    12
registration_days_before      15
previous_events_registered    10
previous_events_attended      14
club_member                    8
event_day                      9
event_time                    11
travel_distance_km            14
attended                       5
dtype: int64

Exact duplicate rows: 7
Duplicate student_id rows: 8

Dtypes:
student_id                     object
event_type                     object
registration_days_before      float64
previous_events_registered    float64
previous_events_attended      float64
club_member                    object
event_day                      object
event_time                     object
travel_distance_km            float64
attended                      float64
dtype: object


In [108]:

quality_report(test_raw, 'TEST (raw)')


QUALITY REPORT: TEST (raw)   shape=(100, 9)

Missing values:
event_type                    4
registration_days_before      3
previous_events_registered    3
previous_events_attended      1
club_member                   3
event_day                     3
event_time                    4
travel_distance_km            2
dtype: int64

Exact duplicate rows: 0
Duplicate student_id rows: 0

Dtypes:
student_id                     object
event_type                     object
registration_days_before      float64
previous_events_registered    float64
previous_events_attended      float64
club_member                    object
event_day                      object
event_time                     object
travel_distance_km            float64
dtype: object


In [109]:
for col in ['event_type', 'club_member', 'event_day']:
    print(col, '->', sorted(train_raw[col].dropna().unique().tolist()))


event_type -> ['Competition', 'HACKATHON', 'Hackathon', 'Social', 'Talk', 'Workshop', 'workshop']
club_member -> ['No', 'YES', 'Yes', 'yes']
event_day -> ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']


In [110]:
# Impossible / illogical values
print('Negative registration_days_before:', (train_raw['registration_days_before'] < 0).sum())
bad_logic = train_raw['previous_events_attended'] > train_raw['previous_events_registered']
print('Rows where attended > registered (impossible):', bad_logic.sum())
print('registration_days_before distribution (top 10):')
print(train_raw['registration_days_before'].sort_values(ascending=False).head(10).values)
print('-> Extreme values are treated as missing, not clipped.')
print('travel_distance_km distribution (top 10):')
print(train_raw['travel_distance_km'].sort_values(ascending=False).head(10).values)
print('-> Extreme values are treated as missing, not clipped.')

Negative registration_days_before: 2
Rows where attended > registered (impossible): 3
registration_days_before distribution (top 10):
[60. 45. 14. 14. 14. 14. 14. 14. 14. 14.]
-> Extreme values are treated as missing, not clipped.
travel_distance_km distribution (top 10):
[120.   75.   60.5  45.   35.5  25.   25.   25.   21.5  20.4]
-> Extreme values are treated as missing, not clipped.


## Cleaning Functions

Each fix operates on a copy. We keep rows wherever possible: suspicious feature values are converted to `NaN` and imputed later. Train-learned imputation values are reused for test.

In [111]:
def standardize_categoricals(df, cols=('event_type', 'club_member', 'event_day', 'event_time')):
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        df[col] = df[col].astype(str).str.strip().str.title()
        df.loc[df[col].isin(['Nan', 'None', '']), col] = np.nan
    if 'club_member' in df.columns:
        df['club_member'] = df['club_member'].map({'Yes': 1, 'No': 0})
    if 'event_time' in df.columns:
        df['event_time'] = pd.to_datetime(
            df['event_time'], format='%H:%M', errors='coerce'
        ).dt.hour
    return df

In [112]:
def fix_impossible_values(df):
    df = df.copy()
    if 'registration_days_before' in df.columns:
        df.loc[df['registration_days_before'] < 0, 'registration_days_before'] = np.nan
    if {'previous_events_registered', 'previous_events_attended'}.issubset(df.columns):
        bad = df['previous_events_attended'] > df['previous_events_registered']
        # We cannot know which value is wrong, so do not invent a swap.
        df.loc[bad, ['previous_events_registered', 'previous_events_attended']] = np.nan
    return df

In [113]:
def mark_outliers_as_missing(df, bounds):
    df = df.copy()
    for col, (lo, hi) in bounds.items():
        if col not in df.columns:
            continue
        mask = (df[col] < lo) | (df[col] > hi)
        df.loc[mask, col] = np.nan
    return df


def learn_outlier_bounds(df, cols, iqr_factor=3):
    bounds = {}
    for col in cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (max(0, q1 - iqr_factor * iqr), q3 + iqr_factor * iqr)
    return bounds

In [114]:
def deduplicate(df, id_col='student_id'):
    df = df.copy()
    before = len(df)
    df = df.drop_duplicates()
    exact_dropped = before - len(df)
    if id_col in df.columns:
        conflicts = df[df[id_col].duplicated(keep=False)].sort_values(id_col)
        if len(conflicts):
            print(f'  repeated {id_col} rows kept: {len(conflicts)}')
    print(f'  dropped {exact_dropped} exact duplicate rows; no rows dropped for repeated {id_col}')
    return df.reset_index(drop=True)

In [115]:

def learn_impute_values(df, numeric_cols, categorical_cols):
    fill = {}
    for col in numeric_cols:
        fill[col] = df[col].median()
    for col in categorical_cols:
        fill[col] = df[col].mode(dropna=True).iloc[0]
    return fill


def impute_missing(df, fill_values):
    df = df.copy()
    for col, val in fill_values.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)
    return df


## `DataCleaner`: fit on train, transform train & test identically



In [116]:
NUMERIC_COLS = ['registration_days_before', 'previous_events_registered', 'previous_events_attended', 'travel_distance_km']
CATEGORICAL_COLS = ['event_type', 'club_member', 'event_day', 'event_time']
OUTLIER_COLS = ['registration_days_before', 'travel_distance_km']

class DataCleaner:
    def __init__(self):
        self.outlier_bounds_ = None
        self.fill_values_ = None
    def fit(self, df):
        tmp = standardize_categoricals(df)
        tmp = fix_impossible_values(tmp)
        tmp = deduplicate(tmp)
        self.outlier_bounds_ = learn_outlier_bounds(tmp, OUTLIER_COLS)
        tmp = mark_outliers_as_missing(tmp, self.outlier_bounds_)
        self.fill_values_ = learn_impute_values(tmp, NUMERIC_COLS, CATEGORICAL_COLS)
        return self
    def transform(self, df, is_train=False, target_col='attended'):
        assert self.outlier_bounds_ is not None, 'call .fit() on train first'
        df = standardize_categoricals(df)
        df = fix_impossible_values(df)
        df = deduplicate(df)
        df = mark_outliers_as_missing(df, self.outlier_bounds_)
        df = impute_missing(df, self.fill_values_)
        if is_train:
            df[target_col] = pd.to_numeric(df[target_col], errors='coerce').astype('Int64')
        for col in NUMERIC_COLS:
            if col in df.columns and col != 'travel_distance_km':
                df[col] = df[col].round().astype(int)
        return df.reset_index(drop=True)
    def fit_transform(self, df, is_train=True, target_col='attended'):
        self.fit(df)
        return self.transform(df, is_train=is_train, target_col=target_col)

In [117]:
cleaner = DataCleaner()
print('Fitting on train...')
train_clean = cleaner.fit_transform(train_raw, is_train=True)
print('Learned outlier bounds:', cleaner.outlier_bounds_)
print('Learned fill values:', cleaner.fill_values_)
print('Transforming test with the SAME learned parameters...')
test_clean = cleaner.transform(test_raw, is_train=False)

# The target must never contain NaN in the modeling dataset.
train_unlabeled = train_clean[train_clean['attended'].isna()].copy()
train_clean = train_clean[train_clean['attended'].notna()].copy()
train_clean['attended'] = train_clean['attended'].astype(int)

print('train_clean shape:', train_clean.shape)
print('unlabeled rows kept separately:', train_unlabeled.shape)
print('test_clean shape:', test_clean.shape)

Fitting on train...
  repeated student_id rows kept: 2
  dropped 7 exact duplicate rows; no rows dropped for repeated student_id
  repeated student_id rows kept: 2
  dropped 7 exact duplicate rows; no rows dropped for repeated student_id
Learned outlier bounds: {'registration_days_before': (0, 32.0), 'travel_distance_km': (0, 26.800000000000004)}
Learned fill values: {'registration_days_before': 7.0, 'previous_events_registered': 4.0, 'previous_events_attended': 3.0, 'travel_distance_km': 5.5, 'event_type': 'Workshop', 'club_member': np.float64(1.0), 'event_day': 'Saturday', 'event_time': np.float64(18.0)}
Transforming test with the SAME learned parameters...
  dropped 0 exact duplicate rows; no rows dropped for repeated student_id
train_clean shape: (496, 10)
unlabeled rows kept separately: (5, 10)
test_clean shape: (100, 9)


## Post-Cleaning Quality Report

The cleaned data keeps rows except exact duplicate records. Suspicious feature values are converted to missing and imputed; rows with missing target labels are preserved in `train_unlabeled`.

In [118]:

quality_report(train_clean, 'TRAIN (cleaned)')


QUALITY REPORT: TRAIN (cleaned)   shape=(496, 10)

Missing values:
  none

Exact duplicate rows: 0
Duplicate student_id rows: 1

Dtypes:
student_id                     object
event_type                     object
registration_days_before        int64
previous_events_registered      int64
previous_events_attended        int64
club_member                   float64
event_day                      object
event_time                    float64
travel_distance_km            float64
attended                        int64
dtype: object


In [119]:

quality_report(test_clean, 'TEST (cleaned)')


QUALITY REPORT: TEST (cleaned)   shape=(100, 9)

Missing values:
  none

Exact duplicate rows: 0
Duplicate student_id rows: 0

Dtypes:
student_id                     object
event_type                     object
registration_days_before        int64
previous_events_registered      int64
previous_events_attended        int64
club_member                   float64
event_day                      object
event_time                    float64
travel_distance_km            float64
dtype: object


In [120]:
# Spot-check: category values and logic
for col in ['event_type', 'club_member', 'event_day', 'event_time']:
    print(col, '->', sorted(train_clean[col].dropna().unique().tolist()))
print('registration_days_before range:', train_clean['registration_days_before'].min(), '-', train_clean['registration_days_before'].max())
print('travel_distance_km range:', round(train_clean['travel_distance_km'].min(), 1), '-', round(train_clean['travel_distance_km'].max(), 1))
print('previous_events_attended > previous_events_registered rows:', (train_clean['previous_events_attended'] > train_clean['previous_events_registered']).sum())
print('missing target rows preserved:', train_clean['attended'].isna().sum())

event_type -> ['Competition', 'Hackathon', 'Social', 'Talk', 'Workshop']
club_member -> [0.0, 1.0]
event_day -> ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
event_time -> [9.0, 11.0, 14.0, 16.0, 18.0]
registration_days_before range: 0 - 14
travel_distance_km range: 0.5 - 25.0
previous_events_attended > previous_events_registered rows: 7
missing target rows preserved: 0


In [121]:
train_clean.to_csv('train_clean.csv', index=False)
train_unlabeled.to_csv('train_unlabeled.csv', index=False)
test_clean.to_csv('test_clean.csv', index=False)
print('Saved train_clean.csv', train_clean.shape)
print('Saved train_unlabeled.csv', train_unlabeled.shape)
print('Saved test_clean.csv', test_clean.shape)

Saved train_clean.csv (496, 10)
Saved train_unlabeled.csv (5, 10)
Saved test_clean.csv (100, 9)


In [122]:
train_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 496 entries, 0 to 500
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  496 non-null    object 
 1   event_type                  496 non-null    object 
 2   registration_days_before    496 non-null    int64  
 3   previous_events_registered  496 non-null    int64  
 4   previous_events_attended    496 non-null    int64  
 5   club_member                 496 non-null    float64
 6   event_day                   496 non-null    object 
 7   event_time                  496 non-null    float64
 8   travel_distance_km          496 non-null    float64
 9   attended                    496 non-null    int64  
dtypes: float64(3), int64(4), object(3)
memory usage: 42.6+ KB
